##### Lab 7: Model Inferencing using TFLite and ONNX Runtimes
*Suggested time: 35-40  minutes*

Accelerator: CPU 

This lab demonstrates the decoupling of the training and the inference environments. Inferencing requires a light weight runtime environment, which parses and executes the operators of the trained (saved) neural network model.

The goal of this lab is to show you:
- Execute Sci-kit Learn ML models using ONNX runtime
- Inference MobileNet models using TFLite
- Continuous inference of audio using recorded sound

#####  About: 

These examples use trained/saved models from Lab-6.  The first performs inference of an ONNX model trained with Random Forest classifier on Iris dataset.
*Classification_report* is a nifty Sci-kit Learn function to determine precision, recall and f1-score.

##### **Step 1:**

*   Upload and unzip model files and datasets
*   Install ONNXruntime
*   Run ONNX inference on Random Forest (RF) model trained on Iris dataset



In [0]:
%%bash

wget -qq https://edge-ai-doulos.s3.us-west-2.amazonaws.com/Edge-AI-Upload.zip
unzip -qq Edge-AI-Upload.zip


In [0]:
!pip install -q onnxruntime

In [0]:
# Compute the prediction with ONNX Runtime

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import onnxruntime as rt
import numpy
import os

base_dir = os.getcwd()
target_dir =os.path.join(base_dir, 'Model-Files', 'rf_iris.onnx')

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y)

target_names = ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']


sess = rt.InferenceSession(target_dir)
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name
pred_onx = sess.run([label_name], {input_name: X_test.astype(numpy.float32)})[0]
print (pred_onx)

print(classification_report(y_test, pred_onx, target_names=target_names))

##### **Step 2:**

Inference of Iris dataset ONNX model trained using K-means clustering.  Notice the difference in metrics when compared to Random Forest Classifier (in earlier example)

In [0]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import onnxruntime as rt
import numpy
import os

base_dir = os.getcwd()
target_dir =os.path.join(base_dir, 'Model-Files', 'km_iris.onnx')

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y)

target_names = ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']


sess = rt.InferenceSession(target_dir)
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name
pred_onx = sess.run([label_name], {input_name: X_test.astype(numpy.float32)})[0]
print (pred_onx)

print(classification_report(y_test, pred_onx, target_names=target_names))

##### **Step 3:**

Inference using ONNX model and ONNX runtime. Please note that only the runtime is needed for inference, not the entire training stack!

In [0]:
# Compute the prediction with ONNX Runtime
import onnxruntime as rt
from numpy import array
import numpy as np
import os

base_dir = os.getcwd()
target_dir =os.path.join(base_dir, 'Model-Files', 'my_fibonacci_onnx_model.onnx')


x_input= array([21,34,55])
n_steps_in =3


x_input = x_input.reshape((1, n_steps_in))

input_data = np.array(x_input, dtype=np.float32)

sess = rt.InferenceSession(target_dir)
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

pred_onx = sess.run([output_name], {input_name: input_data})

print ('Predicted value from ONNX', pred_onx)

######  **Question:**
Why do we change the shape of x_input using the **reshape** method.

######  **Answer:**
<details>
<summary> See our answer </summary>

  - The input shape is changed to conform with the inference shape format *[batch_size, input]*.  The dimension of the input tensor is increased by one.

</details>

*Review of Python Dictionary:*

TFLite Inference output is a Python list containing dictionaries. An element of this inference list is accessed as follows:

inference_list = [{'name': 'Michael', 'age':50, 'occupation': 'Educator'},{'name': 'Mark', 'age':35, 'occupation': 'Engineer'} ]

a = inference_list[0]['name']

print(a)

**Michael**

In [0]:
inference_list = [{'name': 'Michael', 'age':50, 'occupation': 'Educator'},{'name': 'Mark', 'age':35, 'occupation': 'Engineer'} ]

a = inference_list[0]['name']
print (a)

##### **Step 4:**

Inference of Fibonacci using TFlite model schema and associated runtime.  
Install LiteRT runtime.

In [0]:
!pip install -q ai-edge-litert

In [0]:
from ai_edge_litert.interpreter import Interpreter
import numpy as np
from numpy import array
import os

base_dir = os.getcwd()
target_dir =os.path.join(base_dir, 'Model-Files', 'fibonacci_1.tflite')

interpreter = Interpreter(model_path=str(target_dir))

interpreter.allocate_tensors()

x_input= array([21,34,55])

n_steps_in =3

x_input = x_input.reshape((1, n_steps_in))

input_data = np.array(x_input, dtype=np.float32)

input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]

print ('Input Data', input_data)

interpreter.set_tensor(input_index, input_data)
interpreter.invoke()

prediction= interpreter.get_tensor(output_index)

prediction = np.asarray(prediction)

print ('Predicted value from TFLite is',prediction)

##### **Step 5:**

Understand the pretrained model (mobilenet_v1_1.0_224_quant.tflite) for inferencing. The code below is shown for reference. *It will not run.* When executed, the cell will write a file named label_image.py.

Run inference using TFLite runtime on a pretrained mobilenet.tflite model. Download new image, and change its details in the arguments list. The pretrained model is trained on ImageNet dataset.

In [0]:
%%writefile label_image.py

# Copyright 2018 The TensorFlow Authors. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================
"""label_image for tflite."""

import argparse
import time
import os

import numpy as np
from PIL import Image
from ai_edge_litert.interpreter import Interpreter

base_dir = os.getcwd()
target_dir_image =os.path.join(base_dir, 'Images', 'grace_hopper.bmp')
target_dir_models =os.path.join(base_dir, 'Model-Files', 'mobilenet_v1_1.0_224.tflite')
target_dir_label =os.path.join(base_dir, 'Model-Files', 'ImageNet-labels.txt')

def load_labels(filename):
  with open(filename, 'r') as f:
    return [line.strip() for line in f.readlines()]

if __name__ == '__main__':
  parser = argparse.ArgumentParser()
  parser.add_argument(
      '-i',
      '--image',
      default= target_dir_image,
      help='image to be classified')
  parser.add_argument(
      '-m',
      '--model_file',
      default= target_dir_models,
      help='.tflite model to be executed')
  parser.add_argument(
      '-l',
      '--label_file',
      default= target_dir_label,
      help='name of file containing labels')
  parser.add_argument(
      '--input_mean',
      default=127.5, type=float,
      help='input_mean')
  parser.add_argument(
      '--input_std',
      default=127.5, type=float,
      help='input standard deviation')
  parser.add_argument(
      '--num_threads', default=None, type=int, help='number of threads')
  args = parser.parse_args()

  interpreter = Interpreter(
      model_path=args.model_file, num_threads=args.num_threads)
  interpreter.allocate_tensors()

  input_details = interpreter.get_input_details()
  output_details = interpreter.get_output_details()

  # check the type of the input tensor
  floating_model = input_details[0]['dtype'] == np.float32

  # NxHxWxC, H:1, W:2
  height = input_details[0]['shape'][1]
  width = input_details[0]['shape'][2]
  img = Image.open(args.image).resize((width, height))

  # add extra dim
  input_data = np.expand_dims(img, axis=0)

  if floating_model:
    input_data = (np.float32(input_data) - args.input_mean) / args.input_std

  interpreter.set_tensor(input_details[0]['index'], input_data)

  start_time = time.time()
  interpreter.invoke()
  stop_time = time.time()

  output_data = interpreter.get_tensor(output_details[0]['index'])
  results = np.squeeze(output_data)

  top_k = results.argsort()[-5:][::-1]
  labels = load_labels(args.label_file)
  for i in top_k:
    if floating_model:
      print(f'{results[i]:08.6f}:  {labels[i]}')
    else:
      print(f'{(results[i] / 255.0):08.6f}: {labels[i]}')

  print(f'Inference time: {(stop_time - start_time) * 1000:.3f} ms')

######  **Question:**
Why do we change the shape of the input image tensor in the above code?

######  **Answer:**
<details>
<summary> See our answer </summary>

  - The input shape is changed using the Numpy *expand_dims* method.

</details>

##### **Step 6:**  
Run the image classification model with default parameters for model, image, and labels file.

In [0]:
!python label_image.py

##### Questions:

A. In step 6, how were the image, model and labels file selected?

B. What is the XNNPACK delegate used by TensorFlow Lite in step 6?

##### Answer: 

<details>
<summary> See our answer </summary>

  - These were selected based on the default values mentioned in the argument parser.
  - The [XNNPACK](https://github.com/google/XNNPACK) (Accelerated Neural Network PACKage) is a highly optimized library of neural network inference operators designed for various CPU architectures (ARM, x86, WebAssembly)

</details>

##### **Step 7:** 
Experiment by changing models, input image and labels file.  In the example below, we use our custom trained ESC-11 model with a spectrogram image.

In [0]:
%%bash

MODEL_FILE=$(pwd)/Model-Files/ESC-11-MobileNet.tflite
LABEL_FILE=$(pwd)/Model-Files/sound_labels.txt
IMAGE_FILE=$(pwd)/Images/clock-spec.jpg

python label_image.py \
    --model_file $MODEL_FILE \
    --label_file $LABEL_FILE \
    --image $IMAGE_FILE

##### Experiment with image classification inference:

Use different images and model to check working of above code.  The argument parser enables the user to select a model, labels file and image to classify.

In [3]:
import torch
import torchvision.models as models

# 1. Load your model
model = models.resnet50(pretrained=True)
model.eval()  # Mandatory for inference

# 2. Prepare inputs (example with a random tensor)
# Shape: [Batch, Channels, Height, Width]
inputs = torch.randn(1, 3, 224, 224) 

# 3. Perform inference
with torch.no_grad():
    outputs = model(inputs)  # Or model(**inputs) if using a dict/keyword args

# 4. Process results
probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
print(probabilities)


/ext/venvs/cocalc/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/ext/venvs/cocalc/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/user/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


  0%|          | 0.00/97.8M [00:00<?, ?B/s]

  8%|▊         | 8.12M/97.8M [00:00<00:01, 85.0MB/s]

 36%|███▌      | 35.0M/97.8M [00:00<00:00, 200MB/s] 

 62%|██████▏   | 60.6M/97.8M [00:00<00:00, 231MB/s]

 89%|████████▉ | 87.4M/97.8M [00:00<00:00, 250MB/s]

100%|██████████| 97.8M/97.8M [00:00<00:00, 231MB/s]

tensor([4.6145e-05, 1.6332e-04, 8.9351e-05, 5.3784e-05, 1.7425e-04, 2.5811e-05,
        8.5323e-05, 6.0467e-05, 1.2255e-04, 2.7196e-05, 9.6225e-05, 3.1573e-04,
        1.1459e-04, 1.0222e-04, 3.5718e-05, 5.2844e-05, 4.0922e-05, 6.0816e-05,
        1.9143e-04, 1.5599e-04, 6.0580e-05, 1.3727e-04, 9.3567e-05, 1.7508e-04,
        8.9465e-05, 3.6331e-05, 2.2266e-05, 4.6096e-05, 3.0493e-05, 1.8468e-05,
        1.2604e-05, 1.3158e-04, 5.6099e-06, 1.6144e-05, 6.6591e-05, 1.5761e-05,
        1.5499e-04, 7.6794e-06, 1.4275e-04, 1.6900e-04, 1.1207e-04, 1.1316e-05,
        1.1462e-04, 9.7066e-05, 9.6538e-05, 1.2766e-04, 1.2879e-04, 1.3425e-05,
        4.3347e-04, 1.2201e-05, 4.9414e-04, 7.2660e-06, 2.1733e-05, 5.7391e-05,
        2.6292e-04, 5.7513e-05, 1.2304e-04, 1.1544e-05, 1.5054e-04, 2.3650e-04,
        2.1621e-04, 3.6428e-04, 1.8155e-04, 1.1583e-04, 4.8452e-05, 1.0416e-04,
        4.9955e-05, 7.0131e-05, 7.7742e-05, 5.4992e-04, 4.9865e-05, 9.3083e-04,
        1.3441e-04, 2.2842e-04, 1.2634e-